# Telecom Churn Analysis — Experimental Notebook

This notebook runs the **full ML pipeline** using the production `src/` modules.

**Pipeline:** `load_data → validate → clean_data → feature_engineering → train → evaluate → infer`

In [ ]:
import sys
from pathlib import Path

# Ensure the repo root is on sys.path so `src` is importable
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from src.load_data import load_data
from src.validate import validate_dataframe
from src.clean_data import clean_data
from src.feature_engineering import build_features, FeatureConfig
from src.train import train_model
from src.evaluate import evaluate_model
from src.infer import run_inference
from sklearn.preprocessing import FunctionTransformer

## 1. Load Data

In [ ]:
raw_path = REPO_ROOT / "data" / "raw" / "telecom_churn.csv"
df = load_data(raw_path)
print(f"Shape: {df.shape}")
df.head()

## 2. Validate Raw Data

In [ ]:
REQUIRED_COLUMNS = [
    "AccountWeeks", "DataUsage", "CustServCalls",
    "DayMins", "DayCalls", "MonthlyCharge",
    "OverageFee", "RoamMins", "Churn",
    "ContractRenewal", "DataPlan",
]

validate_dataframe(df, REQUIRED_COLUMNS)
print("Validation passed.")

## 3. Clean Data

In [ ]:
df_clean = clean_data(df)
print(f"Shape after cleaning: {df_clean.shape}")
df_clean.head()

## 4. Feature Engineering

In [ ]:
TARGET_COL = "churn"

cfg = FeatureConfig(
    target_col=TARGET_COL,
    numeric_cols=(
        "accountweeks", "datausage", "custservcalls",
        "daymins", "daycalls", "monthlycharge",
        "overagefee", "roammins",
    ),
    categorical_cols=("contractrenewal", "dataplan"),
)

df_feat = build_features(df_clean, cfg)
print(f"Shape after feature engineering: {df_feat.shape}")
df_feat.head()

## 5. Train Model

In [ ]:
y = df_feat[TARGET_COL]
X = df_feat.drop(columns=[TARGET_COL])

preprocessor = FunctionTransformer()  # features already engineered
MODEL_PATH = str(REPO_ROOT / "models" / "model.pkl")
PROBLEM_TYPE = "classification"

fitted_pipeline, X_test, y_test = train_model(
    X, y, preprocessor, PROBLEM_TYPE, MODEL_PATH
)
print(f"Test set size: {len(X_test)}")

## 6. Evaluate Model

In [ ]:
metric = evaluate_model(fitted_pipeline, X_test, y_test, PROBLEM_TYPE)
print(f"\nWeighted F1 Score: {metric:.4f}")

In [ ]:
# Show the confusion matrix plot that evaluate_model saved
cm_path = REPO_ROOT / "reports" / "figures" / "confusion_matrix.png"
if cm_path.exists():
    img = plt.imread(str(cm_path))
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.imshow(img)
    ax.axis("off")
    ax.set_title("Confusion Matrix (saved by evaluate_model)")
    plt.show()

## 7. Inference

In [ ]:
predictions = run_inference(fitted_pipeline, X_test)
print(f"Predictions shape: {predictions.shape}")
predictions.head(10)

## 8. Summary

The full production pipeline ran end-to-end using the `src/` modules:

| Step | Module | Status |
|------|--------|--------|
| Load | `load_data.py` | Done |
| Validate | `validate.py` | Done |
| Clean | `clean_data.py` | Done |
| Features | `feature_engineering.py` | Done |
| Train | `train.py` | Done |
| Evaluate | `evaluate.py` | Done |
| Infer | `infer.py` | Done |